In [ ]:
# ========== 导入：环境变量、HTTP、展示、OpenAI 兼容客户端 ==========

# 导入标准库 os：读环境变量（如 JINA_API_KEY）
import os
# 导入 requests：HTTP GET（后面走 Jina Reader 代理抓论文页）
import requests
# 从 dotenv 导入 load_dotenv：把 .env 里的密钥读进环境变量，避免把密钥写进代码
from dotenv import load_dotenv
# from scraper import fetch_website_contents #not needed as extracting using jina ai
# 从 IPython.display 导入展示工具：在笔记本里渲染 Markdown 摘要
from IPython.display import Markdown, display
# 从 openai 导入 OpenAI 客户端类：后面用 base_url 指向本地 Ollama
from openai import OpenAI


In [ ]:
# ========== 目标论文 URL（可改成你要摘要的论文页） ==========

# 发给抓取器的完整链接；保留原 URL（含 fragment）以免改变抓取结果
paper_url = 'https://rsisinternational.org/journals/ijrsi/articles/ai-powered-automated-and-portable-device-for-retinal-health-assessment/#:~:text=and%20portable%20solution%20using%20a,It%20is%20a'


In [ ]:
# ========== 拉取本地摘要模型 llama3.2:3b ==========

# shell：下载/更新模型；体积与速度适合本地摘要实验
!ollama pull llama3.2:3b
# or any other model you want to use for summari


In [ ]:
# ========== （可选）删除本地模型以腾出空间 ==========

# 需要时可取消注释：ollama rm 会从本机模型库移除指定模型
#!ollama rm llama3.2:3b # Remove the  model if not working or to free up space


In [ ]:
# ========== 常量 + 指向本地 Ollama 的 OpenAI 兼容客户端 ==========

# 本地模型名：需与 ollama pull / ollama list 中的名字一致
MODEL = "llama3.2:3b"
# Ollama 的 OpenAI 兼容基址：注意带 /v1；默认端口 11434
OLLAMA_BASE_URL = "http://localhost:11434/v1"
# base_url 改写请求目标；api_key 对本地 Ollama 通常任意非空即可
ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')


In [ ]:
# ========== system prompt：要求按固定 Markdown 小节抽取论文信息 ==========

# 整段保留英文：这是发给模型的结构化抽取指令，改译会改变输出格式/行为
system_prompt = """
You are a precise and expert research paper analyst.
Given the scraped content of a research paper webpage, extract and present ONLY the following sections in structured markdown.
Do NOT invent or assume anything not present in the content.

## Paper Title
The full title of the paper.

## Authors
A list of all authors.

## Methodology
How the research was conducted — the approach, experimental setup, and evaluation methods used.

## Components / Technologies Used
All hardware components, software frameworks, machine learning models, libraries, datasets, sensors, or tools mentioned.

## PDF Link
The direct link to the PDF version of the paper if present in the content. Otherwise write: *Not found in content.*

## References
A numbered list of the key references cited in the paper.

## Research Gap
The specific gap, limitation, or unsolved problem in existing research that this paper identifies and addresses.

Respond in clean markdown. Do not wrap the markdown in a code block.
"""


In [ ]:
# ========== user prompt 前缀：后面会拼上抓取到的网页正文 ==========

# 英文前缀保留：告诉模型「下面是抓取内容，请严格按 system 指令抽取」
user_prompt_prefix = """
Below is the scraped content of a research paper webpage.
Extract and structure all the requested information exactly as instructed in the system prompt.

---

"""


In [ ]:
# ========== messages_for：把 system +（前缀+正文）收成 Chat messages ==========

def messages_for(website):
    # website 这里实际是抓取后的纯文本内容（字符串），不是 Website 对象
    return [
        {"role": "system", "content": system_prompt},
        # user：前缀说明 + 论文页全文，供模型抽取字段
        {"role": "user", "content": user_prompt_prefix + website}
    ]


In [ ]:
# ========== fetch_website_contents：经 Jina Reader 代理抓取可读正文 ==========

def fetch_website_contents(url):
    # 加载 .env：读入 JINA_API_KEY 等，避免把密钥写进笔记本
    load_dotenv()
    # 从环境变量取 Jina API Key（Environment Variables）
    api_key = os.getenv("JINA_API_KEY")

    # Jina Reader：在目标 URL 前加 https://r.jina.ai/ 即可拿清洗后的文本
    jina_url = f"https://r.jina.ai/{url}"
    headers = {
        # Bearer 鉴权；密钥来自环境变量
        "Authorization": f"Bearer {api_key}",
        # 让 Jina 用浏览器引擎渲染（对动态页更稳）
        "X-Engine": "browser"
    }

    # GET 抓取；失败时 raise_for_status 抛出 HTTPError，便于早发现
    response = requests.get(jina_url, headers=headers)
    response.raise_for_status()
    # 返回可读文本（通常是 Markdown/纯文本混合）
    return response.text


In [ ]:
# ========== summarize：抓取论文页 → 本地 Ollama 抽取结构化摘要 ==========

def summarize(url):
    # 先经 Jina 拿到网页正文
    content = fetch_website_contents(url)
    # 再把正文塞进 messages，调用本地 llama3.2:3b
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=messages_for(content)
    )
    # 返回助手生成的结构化 Markdown
    return response.choices[0].message.content


In [ ]:
# ========== display_summary：生成摘要并在笔记本中渲染 ==========

def display_summary(url):
    # 调用 summarize 拿到模型输出
    summary = summarize(url)
    # 用 Markdown 组件展示（标题/列表更易读）
    display(Markdown(summary))


In [ ]:
# ========== 试跑：对 paper_url 做一次端到端摘要 ==========

# 前提：Ollama 在跑且已 pull MODEL；.env 里有可用的 JINA_API_KEY
display_summary(paper_url)
